# Parte 1 — Baseline RAG

Corpus: 16 papers de las semanas 1 y 2 (inglés), en `corpus/`.

- Embeddings: `bge-m3` en la H200 (VPN GlobalProtect)
- Índice: Qdrant en Docker (`localhost:6333`)
- Generación: `qwen3:1.7b` con Ollama local

In [1]:
import sys
from pathlib import Path

sys.path.append("..")

from pypdf import PdfReader

from rag_pipeline import RagPipeline, EMBEDDING_BACKEND, EMBEDDING_MODEL, QDRANT_URL

print(EMBEDDING_BACKEND, EMBEDDING_MODEL, QDRANT_URL)
pipeline = RagPipeline()
print("modelo servido en la H200:", pipeline.encoder.id_servido)
print("tope:", pipeline.max_seq_tokens)

h200 BAAI/bge-m3 http://localhost:6333


/home/harold/Documents/maestria/primer semestre/IA generativa y agentes/semana 2/taller 2/taller02-rag/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


modelo servido en la H200: bge-m3:latest
tope: 8192


## 1.1 — Ingesta

In [2]:
CORPUS = Path("../corpus")
chunks = pipeline.ingest(CORPUS, chunk_tokens=512, overlap_tokens=102)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (30809 > 8192). Running this sequence through the model will result in indexing errors


ingesta: bai-2022-constitutional-ai.pdf: 34 página(s), 94294 caracteres útiles → 75 fragmento(s)
ingesta: blei-2003-lda.pdf: 30 página(s), 59953 caracteres útiles → 57 fragmento(s)
ingesta: brown-2020-gpt3.pdf: 75 página(s), 186342 caracteres útiles → 156 fragmento(s)
ingesta: doersch-2016-tutorial-vae.pdf: 23 página(s), 36740 caracteres útiles → 33 fragmento(s)
ingesta: edge-2024-graphrag.pdf: 26 página(s), 70385 caracteres útiles → 60 fragmento(s)
ingesta: es-2023-ragas.pdf: 8 página(s), 25583 caracteres útiles → 21 fragmento(s)
ingesta: gao-2023-rag-survey.pdf: 21 página(s), 86641 caracteres útiles → 78 fragmento(s)
ingesta: kingma-2013-vae.pdf: 14 página(s), 31786 caracteres útiles → 31 fragmento(s)
ingesta: lewis-2020-rag.pdf: 19 página(s), 55094 caracteres útiles → 47 fragmento(s)
ingesta: ng-jordan-2001-discriminative-vs-generative.pdf: 8 página(s), 18996 caracteres útiles → 19 fragmento(s)
ingesta: ouyang-2022-instructgpt.pdf: 68 página(s), 143167 caracteres útiles → 118 fragme

Páginas con menos de 200 letras y números (posibles figuras o tablas que se pierden):

In [3]:
for pdf in sorted(CORPUS.glob("*.pdf")):
    for n, pagina in enumerate(PdfReader(str(pdf)).pages, start=1):
        texto = pagina.extract_text() or ""
        utiles = sum(c.isalnum() for c in texto)
        if utiles < 200:
            print(f"{pdf.name} pág {n}: {utiles} -> {texto.strip()[:80]!r}")

brown-2020-gpt3.pdf pág 64: 102 -> 'Figure H.1: All results for all SuperGLUE tasks.\nFigure H.2: Results for SAT tas'
brown-2020-gpt3.pdf pág 65: 88 -> 'Figure H.4: All results for all Arithmetic tasks.\nFigure H.5: All results for al'
brown-2020-gpt3.pdf pág 66: 165 -> 'Figure H.6: All results for all Common Sense Reasoning tasks.\nFigure H.7: All re'
brown-2020-gpt3.pdf pág 67: 81 -> 'Figure H.10: All results for all Scramble tasks.\nFigure H.11: All results for al'
brown-2020-gpt3.pdf pág 75: 171 -> '[ZSW+19b] Daniel M. Ziegler, Nisan Stiennon, Jeffrey Wu, Tom B. Brown, Alec Radf'
rag-anything-2025.pdf pág 15: 199 -> 'RAG-ANYTHING: ALL-IN-ONE RAG FRAMEWORK\nFigure 7: Vision analysis prompt for cont'


Ejemplo de encabezado que se repite en cada página y queda mezclado con el texto:

In [4]:
encabezado = "RAG-ANYTHING: ALL-IN-ONE RAG FRAMEWORK"
con_encabezado = [c for c in chunks if encabezado in c.text]
print(len(con_encabezado), "fragmentos lo contienen")

texto = con_encabezado[3].text
i = texto.index(encabezado)
print(texto[i - 200:i + 150])

22 fragmentos lo contienen
r performance gains become increasingly significant as content length increases. We observe particularly pronounced advantages on long-context materials. This validates our core hypothesis 2 [page=3] RAG-ANYTHING: ALL-IN-ONE RAG FRAMEWORK that dual-graph construction and cross-modal hybrid retrieval are essential for handling complex multimodal mat


Ejemplo de una tabla después de la ingesta (tabla D.1 de GPT-3):

In [ ]:
tabla = next(c for c in chunks if c.id == "brown-2020-gpt3-0100")
print(tabla.text[:300])

1.0 GPT-3 2.7B 5.52E+01 4.77E+21 2,650 300 6 3 2 1.0 GPT-3 6.7B 1.39E+02 1.20E+22 6,660 300 6 3 2 1.0 GPT-3 13B 2.68E+02 2.31E+22 12,850 300 6 3 2 1.0 GPT-3 175B 3.64E+03 3.14E+23 174,600 300 6 3 2 1.0 Table D.1: Starting from the right hand side and moving left, we begin with the number of training


**Pregunta:** documenta qué se pierde o se degrada (tablas, encabezados, pies de página) y qué documentos, si alguno, la ingesta rechazó por no tener texto útil.

La ingesta no rechazó ningún documento: los 16 tienen texto y el más corto (`ng-jordan-2001`) tiene 18.996 caracteres útiles, muy por encima del límite de 200. Lo que se pierde son las figuras: en `brown-2020-gpt3` las páginas 64 a 67 son gráficos de resultados y solo queda el título de cada figura, y pasa lo mismo en la página 15 de `rag-anything-2025`. Los encabezados y números de página no se pierden pero quedan mezclados con el texto: en `rag-anything-2025` el título del paper aparece en 22 fragmentos, en medio de una oración junto con el número de página y la marca `[page=3]`. Las tablas se aplanan: quedan como una fila de números sin columnas, como la tabla D.1 de GPT-3.

## 1.2 — Fragmentación

In [5]:
tok = pipeline.encoder.tokenizer
largos = [len(tok(c.text, add_special_tokens=False)["input_ids"]) for c in chunks]

print("parámetros:", chunks[0].metadata)
print("fragmentos:", len(chunks))
print("tokens por fragmento: min", min(largos), "max", max(largos))

parámetros: {'source_path': '../corpus/bai-2022-constitutional-ai.pdf', 'strategy': 'fixed_size', 'unit': 'tokens', 'chunk_tokens': 512, 'overlap_tokens': 102, 'max_seq_tokens': 8192}
fragmentos: 948
tokens por fragmento: min 61 max 513


**Pregunta:** ¿qué `chunk_tokens` y `overlap_tokens` usas y por qué ese tamaño sirve a tus preguntas?

Se usa `chunk_tokens=512` y `overlap_tokens=102`, que dan 948 fragmentos de entre 61 y 513 tokens. Las preguntas sobre este corpus piden datos puntuales de un paper (un número, una definición, una lista de métricas) que normalmente están en uno o dos párrafos, y 512 tokens es más o menos eso; el solapamiento de 102 hace que si una definición queda cortada al final de un fragmento, aparezca completa en el siguiente. El máximo sale 513 y no 512 porque al pasar los tokens a texto y volver a contarlos a veces sale un token más, pero sigue muy lejos del tope de 8192.

## 1.3 — Embedding e indexación

Fila `embed_local_multilingue`: `BAAI/bge-m3`, 1024 dimensiones, 8192 tokens, multilingüe, verificado 2026-08-27.

In [6]:
pipeline.index(chunks)

info = pipeline.client.get_collection(pipeline.collection)
print("colección:", pipeline.collection)
print("puntos:", pipeline.client.count(pipeline.collection).count)
print("dimensión:", info.config.params.vectors.size)

colección: mmia6013_rag
puntos: 948
dimensión: 1024


El índice se construyó con `bge-m3` (fila `embed_local_multilingue`, verificado 2026-08-27) y solo se consulta con ese mismo modelo.

## 1.4 — Recuperación

In [7]:
preguntas = [
    "How many parameters does the largest GPT-3 model have?",
    "What are the parametric and non-parametric memories in RAG?",
    "Which metrics does RAGAS use to evaluate a RAG system?",
]

for pregunta in preguntas:
    print(pregunta)
    for hit in pipeline.retrieve(pregunta, top_k=5):
        print(f"  {hit['score']:.3f}  {hit['document']:<45} {hit['chunk_id']}")
    print()

How many parameters does the largest GPT-3 model have?
  0.599  brown-2020-gpt3.pdf                           brown-2020-gpt3-0017
  0.588  brown-2020-gpt3.pdf                           brown-2020-gpt3-0039
  0.578  brown-2020-gpt3.pdf                           brown-2020-gpt3-0040
  0.561  brown-2020-gpt3.pdf                           brown-2020-gpt3-0100
  0.559  ouyang-2022-instructgpt.pdf                   ouyang-2022-instructgpt-0004

What are the parametric and non-parametric memories in RAG?
  0.639  lewis-2020-rag.pdf                            lewis-2020-rag-0022
  0.629  lewis-2020-rag.pdf                            lewis-2020-rag-0003
  0.596  lewis-2020-rag.pdf                            lewis-2020-rag-0000
  0.585  lewis-2020-rag.pdf                            lewis-2020-rag-0001
  0.578  lewis-2020-rag.pdf                            lewis-2020-rag-0006

Which metrics does RAGAS use to evaluate a RAG system?
  0.656  gao-2023-rag-survey.pdf                       gao-2023-r

## 1.5 — Generación

In [8]:
print(pipeline.build_prompt("<pregunta>", [{"document": "<doc>", "chunk_id": "<id>", "text": "<fragmento>"}]))

Responde usando solo el contexto recuperado.
Si el contexto no contiene la respuesta, di exactamente: "El corpus no contiene información suficiente."

Contexto:
[1] <doc> / <id>
<fragmento>

Pregunta: <pregunta>
Respuesta:


In [9]:
for pregunta in preguntas:
    salida = pipeline.answer(pregunta, top_k=5)
    print(pregunta)
    print(f"[{salida['generator']}] abstuvo={salida['abstained']}")
    print(salida["answer"])
    print()

How many parameters does the largest GPT-3 model have?
[ollama:qwen3:1.7b] abstuvo=False
El corpus menciona que el modelo más grande es el GPT-3 con 175 bilones de parámetros (175B). La respuesta es: 175B.

What are the parametric and non-parametric memories in RAG?
[ollama:qwen3:1.7b] abstuvo=False
Los memorias paramétricas y no paramétricas en RAG se describen como sigue:  
- **Memoria paramétrica**: Es un modelo seq2seq preentrenado, como BART, que se utiliza para generar salida basada en documentos recuperados.  
- **Memoria no paramétrica**: Es un índice denso de Wikipedia accedido mediante un retriever preentrenado, que permite acceder a información no almacenada en los parámetros del modelo.  

Estas definiciones se encuentran explicitadas en el contexto proporcionado.

Which metrics does RAGAS use to evaluate a RAG system?
[ollama:qwen3:1.7b] abstuvo=False
El corpus menciona que RAGAS utiliza varios métricos como **Hit Rate, MRR, NDCG, EM, Recall, Precision, R-Rate, Cosine Simi

**Observación sobre las respuestas:** En la respuesta de GPT-3 el dato es correcto (el paper dice *175 billion parameters*), pero qwen lo tradujo como "175 bilones". En inglés *billion* es mil millones, mientras que según la RAE (Diccionario panhispánico de dudas) en español un billón es un millón de millones; usar billón como mil millones es común en el español de Estados Unidos, pero no se recomienda. Lo correcto es 175 mil millones de parámetros.